
# Optimisation manuelle (descente de gradient en PyTorch)

**Objectifs de la séance.**
- Étendre le code de la séance précédente à la descente de gradient stochastique, et observer l'effet sur la variance du gradient et la vitesse de convergence.
- Observer l'effet du *learning rate* (trop grand, trop petit, bon compromis).
- Découvrir et utiliser `torch.optim`, en particulier l'optimiseur Adam.


In [ ]:

import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)


## Point de départ : reprise de la séance 1

Voici le code exact sur lequel s'est terminée la séance 1 (ne modifiez pas la génération du jeu de données : vous devez retrouver les mêmes valeurs cibles $w^\star=-3$, $b^\star=1.5$).

In [ ]:

X = torch.rand(100, 1)
# w* = -3, b* = 1.5
y = -3.0 * X + 1.5 + 0.4 * torch.randn(X.size())

def mse(X, y, w, b):
    y_pred = w * X + b
    return torch.mean((y_pred - y) ** 2)

plt.scatter(X.numpy(), y.numpy())
plt.xlabel("X"); plt.ylabel("y")


## Partie 1 — Terminer la régression *full batch*

**Question 1.1.** Reprenez la boucle de descente de gradient *full batch* codée à la séance 1 : initialisez `w` et `b` à 0 (tenseurs scalaires `requires_grad=True`), puis itérez pendant `n_iter = 1000` pas avec `eta = 0.1`. Stockez la valeur de la perte à chaque itération dans une liste `losses_full` (elle servira aux comparaisons de la partie 2).

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

eta = 0.1
n_iter = 1000
losses_full = []

for i in range(n_iter):
    # TODO : calculer la perte, la stocker dans losses_full, déclencher le calcul du gradient,
    #        mettre à jour w et b (dans un bloc torch.no_grad()), puis remettre les gradients à zéro
    loss = mse(X, y, w, b)
    losses_full.append(loss.item())

    loss.backward()

    with torch.no_grad():
        w -= eta * w.grad
        b -= eta * b.grad

    w.grad.zero_()
    b.grad.zero_()


**Question 1.2.** Affichez `losses_full` en fonction du nombre d'itérations, et affichez les valeurs finales de `w` et `b`. Sont-elles proches de $w^\star=-3$ et $b^\star=1.5$ ?

In [ ]:

plt.plot(losses_full)
plt.xlabel("itération"); plt.ylabel("MSE")
plt.title("Descente de gradient full batch")
plt.show()

print("w appris :", w.item(), " (attendu : -3)")
print("b appris :", b.item(), " (attendu : 1.5)")


## Partie 2 — Descente de gradient vs Descente de gradient stochastique

Jusqu'ici, chaque pas de gradient utilisait l'intégralité des 100 exemples. On va maintenant comparer deux façons de calculer le gradient :

- **full batch** : le gradient est calculé sur tout le jeu de données à chaque pas (ce que vous venez de faire) ;
- **stochastique** : le gradient est calculé sur des mini-batchs : le gradient est donc bruité, mais les mises à jour sont plus fréquentes.

Pour comparer sur un pied d'égalité, on raisonne en _epochs_ : une _epoch_ = un passage complet sur les 100 exemples, qu'il soit fait en 1 pas (full batch) ou en 10 pas (mini-batch de taille 10).

**Question 2.1.** Écrivez une fonction `gradient_descent(X, y, batch_size, n_epochs, eta)` qui réinitialise `w` et `b` à 0, puis effectue `n_epochs` _epochs_ de descente de gradient avec la taille de batch demandée (à chaque _epoch_, mélangez les indices avec `torch.randperm(len(X))`, puis parcourez les mini-batchs consécutifs de cette permutation). La fonction doit renvoyer `w`, `b`, et la liste des pertes calculées sur **l'ensemble** du jeu de données à la fin de chaque _epoch_ (pas seulement sur le dernier mini-batch), pour pouvoir comparer les courbes entre elles.

Attention : lorsqu'on stocke la valeur de la perte à la fin de chaque _epoch_, on ne veut pas retenir l'ensemble du graphe de calcul : utilisez `.item()` pour ne stocker que la valeur scalaire de la perte.

In [ ]:
def gradient_descent(X, y, batch_size, n_epochs, eta):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses = []

    for epoch in range(n_epochs):
        # TODO : mélanger les indices avec torch.randperm(len(X))
        perm = torch.randperm(len(X))

        # TODO : parcourir les mini-batchs consécutifs de taille batch_size et faire, pour chacun,
        #        un pas de descente de gradient (perte, backward, mise à jour, remise à zéro)
        for start in range(0, len(X), batch_size):
            idx = perm[start:start + batch_size]
            xb, yb = X[idx], y[idx]

            loss = mse(xb, yb, w, b)
            loss.backward()

            with torch.no_grad():
                w -= eta * w.grad
                b -= eta * b.grad

            w.grad.zero_()
            b.grad.zero_()

        with torch.no_grad():
            losses.append(mse(X, y, w, b).item())  # TODO : calculer la perte sur TOUT le dataset

    return w, b, losses


**Question 2.2.** Appelez `gradient_descent` avec `batch_size` égal à 100 (full batch) et 10 (mini-batch) pour `n_epochs = 30` et `eta = 0.1`. Tracez les deux courbes de perte (par epoch) sur un même graphique.

In [ ]:
_, _, losses_full_ep = gradient_descent(X, y, batch_size=100, n_epochs=30, eta=0.1)
_, _, losses_mini_ep = gradient_descent(X, y, batch_size=10, n_epochs=30, eta=0.1)

plt.plot(losses_full_ep, label="full batch (100)")
plt.plot(losses_mini_ep, label="mini-batch (10)")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend()
plt.title("Full batch vs mini-batch")
plt.show()


**Questions.**
- Quelle courbe est la plus « bruitée » d'une epoch à l'autre ? Pourquoi ?
- Laquelle converge le plus vite *en nombre d'epochs* ?

_La courbe stochastique est la plus bruitée d'une epoch à l'autre : l'estimateur du gradient est bruité. En nombre d'epochs, mini-batch progresse souvent plus vite au début (beaucoup plus de mises à jour par epoch)._


## Partie 3 — Effet du learning rate

**Question 3.1.** En reprenant la version mini-batch (`batch_size=10`) de la partie 2, comparez plusieurs valeurs de `eta` : une très petite (ex. 0.001), une « raisonnable » (ex. 0.1) et une trop grande (ex. 2.0 ou plus). Tracez les courbes de perte correspondantes sur un même graphique (`n_epochs=30`).

In [ ]:
for eta_try in [0.001, 0.1, 2.0]:
    _, _, losses_lr = gradient_descent(X, y, batch_size=10, n_epochs=30, eta=eta_try)
    plt.plot(losses_lr, label=f"eta={eta_try}")

plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend()
plt.title("Effet du learning rate (batch_size=10)")
plt.show()


**Questions.** Que se passe-t-il avec un learning rate trop grand ? Trop petit ? Ce compromis sera au cœur des séances suivantes (Adam, entre autres, essaie d'automatiser ce réglage).

_Avec un learning rate trop grand (2.0), la perte diverge ou oscille violemment au lieu de diminuer. Avec un learning rate trop petit (0.001), la perte diminue mais très lentement. `eta=0.1` offre ici un bon compromis. Adam (question suivante) automatise en partie ce réglage en adaptant le pas par paramètre._


## Partie 4 — `torch.optim` et Adam

Jusqu'ici, la mise à jour `w -= eta * w.grad` était écrite à la main. PyTorch fournit un module `torch.optim` qui encapsule cette logique : on lui donne les paramètres à optimiser, et il s'occupe de la mise à jour (et gère plus de subtilités qu'une simple descente de gradient, comme on va le voir avec Adam).

**Question 4.1.** Réécrivez la fonction de la partie 2, mais en remplaçant la mise à jour manuelle par un `torch.optim.SGD([w, b], lr=0.1)` : à chaque itération, appelez `optimizer.zero_grad()`, `loss.backward()`, puis `optimizer.step()` (plus besoin de `torch.no_grad()` ni de `.grad.zero_()` manuels, l'optimiseur s'en charge). Vérifiez que vous retrouvez (à peu de choses près) les mêmes courbes de perte qu'en partie 1.

In [ ]:
def gradient_descent_optim(X, y, batch_size, n_epochs, eta):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([w, b], lr=eta)
    losses = []

    for epoch in range(n_epochs):
        perm = torch.randperm(len(X))

        for start in range(0, len(X), batch_size):
            idx = perm[start:start + batch_size]
            xb, yb = X[idx], y[idx]

            optimizer.zero_grad()
            loss = mse(xb, yb, w, b)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            losses.append(mse(X, y, w, b).item())

    return w, b, losses


w_sgd, b_sgd, losses_sgd = gradient_descent_optim(
    X, y, batch_size=100, n_epochs=n_iter, eta=0.1
)

plt.plot(losses_full, label="mise à jour manuelle (partie 1)")
plt.plot(losses_sgd, label="torch.optim.SGD")
plt.xlabel("itération / epoch"); plt.ylabel("MSE"); plt.legend()
plt.title("Mise à jour manuelle vs torch.optim.SGD (full batch)")
plt.show()


Adam est un optimiseur adaptatif : il maintient, pour chaque paramètre, une moyenne mobile du gradient (une composante de type *momentum*) et une moyenne mobile du carré du gradient (pour adapter le learning rate paramètre par paramètre). On ne le réimplémente pas : on l'utilise tel quel.

**Question 4.2.** Reprenez la boucle précédente en remplaçant `torch.optim.SGD` par `torch.optim.Adam`. Comparez la courbe de perte obtenue à celle obtenue avec SGD, pour le même learning rate.

In [ ]:
def gradient_descent_adam(X, y, batch_size, n_epochs, eta):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.Adam([w, b], lr=eta)
    losses = []

    for epoch in range(n_epochs):
        perm = torch.randperm(len(X))

        for start in range(0, len(X), batch_size):
            idx = perm[start:start + batch_size]
            xb, yb = X[idx], y[idx]

            optimizer.zero_grad()
            loss = mse(xb, yb, w, b)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            losses.append(mse(X, y, w, b).item())

    return w, b, losses

w_adam, b_adam, losses_adam = gradient_descent_adam(
    X, y, batch_size=100, n_epochs=n_iter, eta=0.1
)

plt.plot(losses_sgd, label="SGD (lr=0.1)")
plt.plot(losses_adam, label="Adam (lr=0.1)")
plt.xlabel("itération / epoch"); plt.ylabel("MSE"); plt.legend()
plt.title("torch.optim.SGD vs torch.optim.Adam")
plt.show()


**Question.** À learning rate identique, Adam est-il plus rapide, plus lent, plus stable que SGD sur ce problème ? Le constat serait-il forcément le même sur un problème plus complexe (paysage de la perte moins « gentil » qu'une régression linéaire) ?

_À learning rate identique, Adam converge en général plus vite et de façon plus stable que SGD sur ce problème (paysage convexe, bien conditionné), car il adapte le pas d'apprentissage paramètre par paramètre et intègre un effet de type momentum. Sur un problème plus complexe (paysage non convexe, mal conditionné, ravins), l'écart entre SGD nu et Adam serait généralement encore plus marqué en faveur d'Adam — ou nécessiterait au minimum d'ajouter du momentum à SGD._


## Bilan

Vous savez maintenant écrire une boucle d'optimisation complète en PyTorch, comparer une descente de gradent avec une descente de gradient stochastique, régler un learning rate, et utiliser `torch.optim`.